# Dataset Audit Validation

## Project
Logistics Operations Power BI

## Purpose

This notebook validates the potential data-quality findings identified by the
automated dataset audit.

The automated audit is treated as a screening mechanism rather than a final
determination of data quality. Each flagged condition is evaluated using:

- Business context
- Data completeness
- Relational integrity
- Dataset structure
- Analytical usability

## Validation Principle

Automated Finding
→ Analyst Investigation
→ Evidence-Based Classification
→ Final Audit Conclusion

## Finding Classifications

Each automated finding is classified as one of:

- Valid Business Condition
- Dataset Limitation
- Data Completeness / Referential Integrity Concern
- Confirmed Data-Quality Issue

## Data Integrity Rule

Raw source files under `data/raw/` are immutable.

No source data is modified, overwritten, deleted, or cleaned during this
audit-validation process.

In [3]:
# ============================================================
# LOAD RAW DATA
# ============================================================

import pandas as pd

# Load the drivers table
drivers = pd.read_csv("../../data/raw/drivers.csv")

print("Drivers table loaded successfully.")
print("Total records:", len(drivers))
print("Columns:", drivers.columns.tolist())

Drivers table loaded successfully.
Total records: 150
Columns: ['driver_id', 'first_name', 'last_name', 'hire_date', 'termination_date', 'license_number', 'license_state', 'date_of_birth', 'home_terminal', 'employment_status', 'cdl_class', 'years_experience']


## 3.1 Driver Employment Status vs. Termination Date

### Audit Finding

The automated dataset audit identified 124 missing values in
`drivers.termination_date`, representing 82.67% of the driver records.

### Validation Question

Are the missing termination dates genuine data-quality issues,
or are they expected for currently active drivers?

### Business Rule

A driver with `employment_status = Active` should normally have
no `termination_date`.

A driver with `employment_status = Terminated` should normally
have a `termination_date`.

### Validation Approach

We will compare `employment_status` against the presence or
absence of `termination_date` to determine whether the missing
values follow the expected business rule.

In [4]:
# ============================================================
# VALIDATION 3.1
# Driver Employment Status vs. Termination Date
# ============================================================

# Create indicators for termination date presence
drivers["termination_date_missing"] = drivers["termination_date"].isna()

# Compare employment status with termination date availability
termination_validation = pd.crosstab(
    drivers["employment_status"],
    drivers["termination_date_missing"]
)

print("Employment status vs. termination date:")
print(termination_validation)

Employment status vs. termination date:
termination_date_missing  False  True 
employment_status                     
Active                        0    124
Terminated                   26      0


In [5]:
# Check for business-rule violations

active_with_termination = drivers[
    (drivers["employment_status"] == "Active") &
    (drivers["termination_date"].notna())
]

terminated_without_termination = drivers[
    (drivers["employment_status"] == "Terminated") &
    (drivers["termination_date"].isna())
]

print("Active drivers with a termination date:",
      len(active_with_termination))

print("Terminated drivers without a termination date:",
      len(terminated_without_termination))

Active drivers with a termination date: 0
Terminated drivers without a termination date: 0


### Investigation Finding

The 124 missing `termination_date` values are associated entirely
with drivers whose `employment_status` is `Active`.

All 26 drivers classified as `Terminated` have a populated
`termination_date`.

No records were found that violate the expected employment-status
and termination-date relationship.

### Final Classification

**Valid Business Condition**

The missing termination dates are not treated as a data-quality
error. They represent active drivers who have not been terminated.

### Data Treatment

No values will be imputed or modified.

The raw `drivers.csv` file remains unchanged.

### Downstream BI Consideration

`employment_status` should be used as the primary field for
current driver employment analysis. `termination_date` can be
used for historical termination analysis where applicable.

## 3.2 Driver CDL Class — Cardinality Validation

### Audit Finding

The automated audit identified `cdl_class` as a low-cardinality
column because it contains only one unique non-null value.

### Validation Question

Does the single observed CDL class indicate a data-quality problem,
or is it simply a characteristic of the available driver population?

### Validation Approach

We will examine the distinct values and their record counts to
determine whether the column contains unexpected, missing, or
inconsistent values.

In [6]:
# ============================================================
# VALIDATION 3.2
# Driver CDL Class
# ============================================================

print("CDL class distribution:")
print(drivers["cdl_class"].value_counts(dropna=False))

print("\nNumber of unique non-null CDL classes:",
      drivers["cdl_class"].nunique())

CDL class distribution:
cdl_class
A    150
Name: count, dtype: int64

Number of unique non-null CDL classes: 1


### Investigation Finding

The `cdl_class` column contains a single non-null value:

- Class A: 150 drivers
- Other CDL classes: 0

There is no evidence from the audit that the values are invalid or
inconsistent.

### Final Classification

**Dataset Limitation**

The dataset represents a driver population where all recorded drivers
have Class A CDL. Because there is no variation in `cdl_class`, the
field cannot support comparative analysis across CDL classes.

### Data Treatment

No values will be modified, recoded, or removed.

The raw `drivers.csv` file remains unchanged.

### Downstream BI Consideration

`cdl_class` may be retained as a descriptive attribute, but it should
not be used for comparative CDL-class analysis or a CDL-class
distribution visual.

In [7]:
# Remove temporary audit-validation column
drivers.drop(columns=["termination_date_missing"], inplace=True)

print("Temporary validation column removed.")

Temporary validation column removed.


## 3.3 Load Status — Cardinality Validation

### Audit Finding

The automated audit identified `load_status` as a low-cardinality
column because only one unique value is present.

### Validation Question

Does the single observed load status indicate a data-quality issue,
or does it reflect the scope of the available dataset?

### Validation Approach

We will examine the distribution of `load_status` and verify the
number of load records represented by the column.

In [8]:
# ============================================================
# VALIDATION 3.3
# Load Status
# ============================================================

# Load the loads table
loads = pd.read_csv("../../data/raw/loads.csv")

print("Loads table loaded successfully.")
print("Total records:", len(loads))

print("\nLoad status distribution:")
print(loads["load_status"].value_counts(dropna=False))

print("\nUnique non-null load statuses:",
      loads["load_status"].nunique())

Loads table loaded successfully.
Total records: 85410

Load status distribution:
load_status
Completed    85410
Name: count, dtype: int64

Unique non-null load statuses: 1


### Investigation Finding

The `load_status` column contains a single non-null value:

- Completed: 85,410 loads
- Other statuses: 0
- Missing statuses: 0

There is no evidence of invalid or inconsistent status values.

### Final Classification

**Dataset Limitation**

The available dataset contains only completed loads. Therefore,
`load_status` does not provide sufficient variation for comparative
load-status analysis.

### Data Treatment

No values will be modified, recoded, or removed.

The raw `loads.csv` file remains unchanged.

### Downstream BI Consideration

`load_status` should not be used for a status-distribution visual
because all records belong to the same category. The limitation
should be documented when defining the dashboard's analytical scope.

## 3.4 Trailer Attributes — Cardinality Validation

### Audit Findings

The automated audit identified two low-cardinality columns in the
`trailers` table:

- `length_feet`
- `status`

### Validation Questions

1. Does `length_feet` contain meaningful variation?
2. Does `status` contain meaningful variation?
3. Do the observed values indicate a data-quality problem or simply
   describe the scope of the dataset?

### Validation Approach

We will examine the value distributions and record counts for both
columns.

In [9]:
# ============================================================
# VALIDATION 3.4
# Trailer Attributes
# ============================================================

# Load the trailers table
trailers = pd.read_csv("../../data/raw/trailers.csv")

print("Trailers table loaded successfully.")
print("Total records:", len(trailers))

print("\nLength distribution:")
print(trailers["length_feet"].value_counts(dropna=False))

print("\nStatus distribution:")
print(trailers["status"].value_counts(dropna=False))

Trailers table loaded successfully.
Total records: 180

Length distribution:
length_feet
53    180
Name: count, dtype: int64

Status distribution:
status
Active    180
Name: count, dtype: int64


### Investigation Findings

#### `length_feet`

All 180 trailer records have `length_feet = 53`.

There is no evidence of invalid or inconsistent values. However,
the absence of variation means trailer-length comparison cannot
be performed using this dataset.

#### `status`

All 180 trailer records have `status = Active`.

There is no evidence of invalid or inconsistent status values.
The dataset therefore does not provide variation for active vs.
inactive trailer analysis.

### Final Classification

**Dataset Limitation**

Both findings represent limitations of the available dataset rather
than confirmed data-quality issues.

### Data Treatment

No values will be modified, recoded, or removed.

The raw `trailers.csv` file remains unchanged.

### Downstream BI Consideration

`length_feet` and `status` may be retained as descriptive attributes,
but they should not be used for comparative visuals because the
dataset contains only one value for each field.

## 3.5 Trip Status — Cardinality Validation

### Audit Finding

The automated audit identified `trip_status` as a low-cardinality
column because only one unique value is present.

### Validation Question

Does the single observed trip status indicate a data-quality issue,
or does it represent the scope of the available trip data?

### Validation Approach

We will examine the distribution of `trip_status` and verify the
number of trip records represented.

In [10]:
# ============================================================
# VALIDATION 3.5
# Trip Status
# ============================================================

# Load the trips table
trips = pd.read_csv("../../data/raw/trips.csv")

print("Trips table loaded successfully.")
print("Total records:", len(trips))

print("\nTrip status distribution:")
print(trips["trip_status"].value_counts(dropna=False))

print("\nUnique non-null trip statuses:",
      trips["trip_status"].nunique())

Trips table loaded successfully.
Total records: 85410

Trip status distribution:
trip_status
Completed    85410
Name: count, dtype: int64

Unique non-null trip statuses: 1


### Investigation Finding

The `trip_status` column contains a single non-null value:

- Completed: 85,410 trips
- Other statuses: 0
- Missing statuses: 0

There is no evidence of invalid or inconsistent status values.

### Final Classification

**Dataset Limitation**

The available dataset contains only completed trips. Therefore,
`trip_status` cannot support comparative analysis of trip lifecycle
states such as completed, delayed, cancelled, or in-transit.

### Data Treatment

No values will be modified, recoded, or removed.

The raw `trips.csv` file remains unchanged.

### Downstream BI Consideration

`trip_status` should not be used to create a trip-status comparison
visual. Operational analysis should instead focus on the available
trip-performance measures such as distance, duration, fuel usage,
MPG, and idle time.

## 3.6 Truck Fuel Type — Cardinality Validation

### Audit Finding

The automated audit identified `fuel_type` as a low-cardinality
column because only one unique value is present.

### Validation Question

Does the single observed fuel type indicate a data-quality issue,
or does it reflect the composition of the available truck fleet?

### Validation Approach

We will examine the distribution of `fuel_type` and verify the
number of truck records represented.

In [11]:
# ============================================================
# VALIDATION 3.6
# Truck Fuel Type
# ============================================================

# Load the trucks table
trucks = pd.read_csv("../../data/raw/trucks.csv")

print("Trucks table loaded successfully.")
print("Total records:", len(trucks))

print("\nFuel type distribution:")
print(trucks["fuel_type"].value_counts(dropna=False))

print("\nUnique non-null fuel types:",
      trucks["fuel_type"].nunique())

Trucks table loaded successfully.
Total records: 120

Fuel type distribution:
fuel_type
Diesel    120
Name: count, dtype: int64

Unique non-null fuel types: 1


### Investigation Finding

The `fuel_type` column contains a single non-null value:

- Diesel: 120 trucks
- Other fuel types: 0
- Missing fuel types: 0

There is no evidence of invalid or inconsistent fuel-type values.

### Final Classification

**Dataset Limitation**

The available truck fleet consists entirely of diesel-powered trucks.
Because there is no fuel-type variation, comparative analysis across
fuel types is not possible.

### Data Treatment

No values will be modified, recoded, or removed.

The raw `trucks.csv` file remains unchanged.

### Downstream BI Consideration

`fuel_type` may be retained as a descriptive fleet attribute, but it
should not be used for comparative fuel-type analysis or a
fuel-type distribution visual.

## 4.1 Fuel Purchase Assignment Completeness

### Audit Findings

The automated audit identified missing assignment identifiers
in the `fuel_purchases` table:

- `truck_id`: 3,880 missing values
- `driver_id`: 3,988 missing values

### Validation Question

Are the missing truck and driver identifiers recoverable through the
available `trip_id` relationship, or do they represent genuine
assignment gaps in the source data?

### Validation Approach

We will:

1. Examine the missingness combinations.
2. Compare `fuel_purchases.trip_id` with `trips.trip_id`.
3. Determine whether missing truck or driver identifiers can be
   recovered from the corresponding trip.
4. Classify the resulting data-quality concern.

In [12]:
# ============================================================
# VALIDATION 4.1
# Fuel Purchase Assignment Completeness
# ============================================================

# Load the fuel purchases table
fuel_purchases = pd.read_csv("../../data/raw/fuel_purchases.csv")

print("Fuel purchases table loaded successfully.")
print("Total records:", len(fuel_purchases))

print("\nMissing assignment identifiers:")
print(
    fuel_purchases[["truck_id", "driver_id"]]
    .isna()
    .sum()
)

Fuel purchases table loaded successfully.
Total records: 196442

Missing assignment identifiers:
truck_id     3880
driver_id    3988
dtype: int64


In [13]:
# ============================================================
# FUEL PURCHASE ASSIGNMENT PATTERNS
# ============================================================

truck_missing = fuel_purchases["truck_id"].isna()
driver_missing = fuel_purchases["driver_id"].isna()

assignment_pattern = pd.DataFrame({
    "truck_missing": truck_missing,
    "driver_missing": driver_missing
})

print("Fuel purchase assignment patterns:")
print(
    assignment_pattern.value_counts()
    .rename("records")
)

Fuel purchase assignment patterns:
truck_missing  driver_missing
False          False             188668
               True                3894
True           False               3786
               True                  94
Name: records, dtype: int64


In [14]:
# ============================================================
# CHECK RECOVERABILITY THROUGH TRIP RELATIONSHIP
# ============================================================

# Select the identifiers needed from the trips table
trip_assignments = trips[
    ["trip_id", "truck_id", "driver_id"]
].copy()

# Join fuel purchases to trips using trip_id
fuel_trip_check = fuel_purchases.merge(
    trip_assignments,
    on="trip_id",
    how="left",
    suffixes=("_fuel", "_trip")
)

# Check missing fuel identifiers where the corresponding trip
# contains the identifier
recoverable_truck = (
    fuel_trip_check["truck_id_fuel"].isna()
    & fuel_trip_check["truck_id_trip"].notna()
)

recoverable_driver = (
    fuel_trip_check["driver_id_fuel"].isna()
    & fuel_trip_check["driver_id_trip"].notna()
)

print(
    "Missing fuel truck_id + trip truck available:",
    recoverable_truck.sum()
)

print(
    "Missing fuel driver_id + trip driver available:",
    recoverable_driver.sum()
)

Missing fuel truck_id + trip truck available: 0
Missing fuel driver_id + trip driver available: 0


In [15]:
# ============================================================
# CHECK FUEL PURCHASE TRIP REFERENCES
# ============================================================

trip_ids = set(trips["trip_id"].dropna())

fuel_trip_exists = fuel_purchases["trip_id"].isin(trip_ids)

print(
    "Fuel purchase records with a matching trip:",
    fuel_trip_exists.sum()
)

print(
    "Fuel purchase records with no matching trip:",
    (~fuel_trip_exists).sum()
)

Fuel purchase records with a matching trip: 196442
Fuel purchase records with no matching trip: 0


In [16]:
# ============================================================
# RECONCILE FUEL PURCHASE IDs WITH TRIP IDs
# ============================================================

print("Fuel purchase records:", len(fuel_trip_check))

print("\nMissingness after trip relationship:")
print(
    fuel_trip_check[
        [
            "truck_id_fuel",
            "driver_id_fuel",
            "truck_id_trip",
            "driver_id_trip"
        ]
    ].isna().sum()
)

print("\nFuel truck missing + trip truck missing:",
      (
          fuel_trip_check["truck_id_fuel"].isna()
          & fuel_trip_check["truck_id_trip"].isna()
      ).sum())

print("Fuel driver missing + trip driver missing:",
      (
          fuel_trip_check["driver_id_fuel"].isna()
          & fuel_trip_check["driver_id_trip"].isna()
      ).sum())

Fuel purchase records: 196442

Missingness after trip relationship:
truck_id_fuel     3880
driver_id_fuel    3988
truck_id_trip     3880
driver_id_trip    3988
dtype: int64

Fuel truck missing + trip truck missing: 3880
Fuel driver missing + trip driver missing: 3988


In [17]:
# ============================================================
# VERIFY MISSINGNESS PROPAGATION
# ============================================================

fuel_missing_truck = fuel_trip_check["truck_id_fuel"].isna()
trip_missing_truck = fuel_trip_check["truck_id_trip"].isna()

fuel_missing_driver = fuel_trip_check["driver_id_fuel"].isna()
trip_missing_driver = fuel_trip_check["driver_id_trip"].isna()

print(
    "Fuel truck missing AND trip truck missing:",
    (fuel_missing_truck & trip_missing_truck).sum()
)

print(
    "Fuel truck missing BUT trip truck available:",
    (fuel_missing_truck & ~trip_missing_truck).sum()
)

print(
    "Fuel driver missing AND trip driver missing:",
    (fuel_missing_driver & trip_missing_driver).sum()
)

print(
    "Fuel driver missing BUT trip driver available:",
    (fuel_missing_driver & ~trip_missing_driver).sum()
)

Fuel truck missing AND trip truck missing: 3880
Fuel truck missing BUT trip truck available: 0
Fuel driver missing AND trip driver missing: 3988
Fuel driver missing BUT trip driver available: 0


### Investigation Findings

The automated audit identified missing assignment identifiers in
`fuel_purchases`:

- `truck_id`: 3,880 missing values
- `driver_id`: 3,988 missing values

Cross-table validation produced the following results:

- All 196,442 fuel purchase records have a matching `trip_id` in
  the `trips` table.
- All 3,880 fuel records with a missing `truck_id` correspond to
  trips where `truck_id` is also missing.
- All 3,988 fuel records with a missing `driver_id` correspond to
  trips where `driver_id` is also missing.
- No missing fuel `truck_id` can be recovered from an available
  trip-level `truck_id`.
- No missing fuel `driver_id` can be recovered from an available
  trip-level `driver_id`.

### Final Classification

**Data Completeness / Referential Assignment Concern**

The `trip_id` relationship is complete, but assignment information
for truck and driver is missing in the underlying trip records and
therefore propagates into the fuel purchase records.

### Data Treatment

No identifiers will be artificially inferred or imputed during the
audit.

The raw `fuel_purchases.csv` and `trips.csv` files remain unchanged.

### Downstream BI Consideration

Fuel analysis involving truck-level or driver-level attribution must
account for these unassigned records.

Fuel metrics should not assume that every fuel purchase can be
attributed to a specific truck or driver.

The missing assignment rates should be considered during data-model
and KPI validation.

## 4.2 Safety Incident Assignment Completeness

### Audit Findings

The automated audit identified one missing `truck_id` and one missing
`driver_id` in the `safety_incidents` table.

### Validation Question

Are these missing identifiers recoverable through the available
`trip_id` relationship, or do they represent genuine assignment gaps?

### Validation Approach

We will:

1. Identify the affected incident records.
2. Verify that their `trip_id` values exist in `trips`.
3. Compare incident-level truck and driver identifiers with the
   corresponding trip-level identifiers.
4. Determine whether the missing identifiers can be recovered.

In [18]:
# ============================================================
# VALIDATION 4.2
# Safety Incident Assignment Completeness
# ============================================================

# Load the safety incidents table
safety_incidents = pd.read_csv(
    "../../data/raw/safety_incidents.csv"
)

print("Safety incidents table loaded successfully.")
print("Total records:", len(safety_incidents))

print("\nMissing assignment identifiers:")
print(
    safety_incidents[["truck_id", "driver_id"]]
    .isna()
    .sum()
)

Safety incidents table loaded successfully.
Total records: 170

Missing assignment identifiers:
truck_id     1
driver_id    1
dtype: int64


In [19]:
# ============================================================
# IDENTIFY INCIDENTS WITH MISSING ASSIGNMENTS
# ============================================================

missing_incident_assignments = safety_incidents[
    safety_incidents["truck_id"].isna()
    | safety_incidents["driver_id"].isna()
]

print("Incidents with missing truck or driver:")
print(
    missing_incident_assignments[
        [
            "incident_id",
            "trip_id",
            "truck_id",
            "driver_id"
        ]
    ].to_string(index=False)
)

Incidents with missing truck or driver:
incident_id      trip_id truck_id driver_id
INC00000060 TRIP00067707      NaN  DRV00101
INC00000086 TRIP00078852 TRK00120       NaN


In [20]:
# ============================================================
# CHECK WHETHER MISSING INCIDENT IDs ARE RECOVERABLE
# ============================================================

incident_trip_check = missing_incident_assignments.merge(
    trips[["trip_id", "truck_id", "driver_id"]],
    on="trip_id",
    how="left",
    suffixes=("_incident", "_trip")
)

print(
    incident_trip_check[
        [
            "incident_id",
            "trip_id",
            "truck_id_incident",
            "driver_id_incident",
            "truck_id_trip",
            "driver_id_trip"
        ]
    ].to_string(index=False)
)

incident_id      trip_id truck_id_incident driver_id_incident truck_id_trip driver_id_trip
INC00000060 TRIP00067707               NaN           DRV00101           NaN       DRV00101
INC00000086 TRIP00078852          TRK00120                NaN      TRK00120            NaN


In [21]:
# ============================================================
# RECOVERABILITY CHECK
# ============================================================

missing_incident_truck_recoverable = (
    incident_trip_check["truck_id_incident"].isna()
    & incident_trip_check["truck_id_trip"].notna()
)

missing_incident_driver_recoverable = (
    incident_trip_check["driver_id_incident"].isna()
    & incident_trip_check["driver_id_trip"].notna()
)

print(
    "Missing incident truck_id recoverable through trip:",
    missing_incident_truck_recoverable.sum()
)

print(
    "Missing incident driver_id recoverable through trip:",
    missing_incident_driver_recoverable.sum()
)

Missing incident truck_id recoverable through trip: 0
Missing incident driver_id recoverable through trip: 0


### Investigation Findings

The automated audit identified:

- 1 missing `truck_id` in `safety_incidents`
- 1 missing `driver_id` in `safety_incidents`

The affected records were traced through their corresponding
`trip_id` values.

Results:

- `INC00000060` has a missing incident-level `truck_id`, and the
  corresponding trip also has a missing `truck_id`.
- `INC00000086` has a missing incident-level `driver_id`, and the
  corresponding trip also has a missing `driver_id`.
- Neither missing identifier can be recovered from the corresponding
  trip record.
- The available truck and driver identifiers are consistent between
  the incident and trip records.

### Final Classification

**Data Completeness / Referential Assignment Concern**

The missing identifiers represent genuine assignment gaps in the
source data. The `trip_id` relationship itself is usable, but the
corresponding assignment information is incomplete.

### Data Treatment

No identifiers will be artificially inferred or imputed.

The raw `safety_incidents.csv` and `trips.csv` files remain unchanged.

### Downstream BI Consideration

Incident analysis involving truck-level or driver-level attribution
must account for these two unassigned incidents.

Incident-level metrics should not assume complete truck and driver
assignment.

## 4.3 Trip Assignment Completeness

### Audit Findings

The automated audit identified missing assignment identifiers in the
`trips` table:

- `driver_id`: 1,714 missing values
- `truck_id`: 1,672 missing values
- `trailer_id`: 1,680 missing values

### Validation Question

How are the missing driver, truck, and trailer assignments distributed
across trip records, and are the missing assignments associated with
specific operational categories?

### Validation Approach

We will:

1. Quantify missingness for each assignment identifier.
2. Examine combinations of missing identifiers.
3. Compare missingness across `load_type`.
4. Compare missingness across `booking_type`.
5. Determine whether the missingness appears concentrated in a
   particular operational segment.

No identifiers will be inferred or modified during this analysis.

In [22]:
# ============================================================
# VALIDATION 4.3
# Trip Assignment Completeness
# ============================================================

print("Trips table already loaded.")
print("Total records:", len(trips))

print("\nMissing assignment identifiers:")
print(
    trips[["driver_id", "truck_id", "trailer_id"]]
    .isna()
    .sum()
)

Trips table already loaded.
Total records: 85410

Missing assignment identifiers:
driver_id     1714
truck_id      1672
trailer_id    1680
dtype: int64


In [23]:
# ============================================================
# TRIP ASSIGNMENT MISSINGNESS PATTERNS
# ============================================================

trip_assignment_pattern = pd.DataFrame({
    "driver_missing": trips["driver_id"].isna(),
    "truck_missing": trips["truck_id"].isna(),
    "trailer_missing": trips["trailer_id"].isna()
})

pattern_counts = trip_assignment_pattern.value_counts()

print("Trip assignment missingness patterns:")
print(pattern_counts)

Trip assignment missingness patterns:
driver_missing  truck_missing  trailer_missing
False           False          False              80458
True            False          False               1636
False           False          True                1603
                True           False               1599
True            False          True                  41
                True           False                 37
False           True           True                  36
Name: count, dtype: int64


In [27]:
# ============================================================
# MISSINGNESS BY LOAD TYPE
# ============================================================

# Load only the columns required from the loads table
loads_lookup = pd.read_csv(
    "../../data/raw/loads.csv",
    usecols=["load_id", "load_type", "booking_type"]
)

# Create the trip assignment missingness pattern
trip_load_analysis = trips[
    ["trip_id", "load_id", "driver_id", "truck_id", "trailer_id"]
].copy()

trip_load_analysis["missing_pattern"] = (
    trip_load_analysis["driver_id"].isna().astype(int).astype(str)
    + trip_load_analysis["truck_id"].isna().astype(int).astype(str)
    + trip_load_analysis["trailer_id"].isna().astype(int).astype(str)
)

# Connect trips to loads through load_id
trip_load_analysis = trip_load_analysis.merge(
    loads_lookup,
    on="load_id",
    how="left"
)

print("Trip assignment missingness by load type:")
print(
    pd.crosstab(
        trip_load_analysis["missing_pattern"],
        trip_load_analysis["load_type"]
    )
)

Trip assignment missingness by load type:
load_type        Dry Van  Refrigerated
missing_pattern                       
000                40001         40457
001                  788           815
010                  798           801
011                   19            17
100                  823           813
101                   17            24
110                   18            19


In [28]:
# ============================================================
# MISSINGNESS BY BOOKING TYPE
# ============================================================

print("Trip assignment missingness by booking type:")
print(
    pd.crosstab(
        trip_load_analysis["missing_pattern"],
        loads_lookup.set_index("load_id")
        .reindex(trip_load_analysis["load_id"])["booking_type"]
    )
)

Trip assignment missingness by booking type:
Empty DataFrame
Columns: []
Index: []


In [29]:
# ============================================================
# MISSINGNESS BY BOOKING TYPE
# ============================================================

print("Trip assignment missingness by booking type:")
print(
    pd.crosstab(
        trip_load_analysis["missing_pattern"],
        trip_load_analysis["booking_type"]
    )
)

Trip assignment missingness by booking type:
booking_type     Contract  Dedicated   Spot
missing_pattern                            
000                 20311      39826  20321
001                   375        839    389
010                   420        801    378
011                    10         21      5
100                   410        810    416
101                     8         21     12
110                    11         19      7


In [30]:
# ============================================================
# MISSING ASSIGNMENT RATE BY BOOKING TYPE
# ============================================================

# Create a flag for whether any trip assignment is missing
trip_load_analysis["any_assignment_missing"] = (
    trip_load_analysis["driver_id"].isna()
    | trip_load_analysis["truck_id"].isna()
    | trip_load_analysis["trailer_id"].isna()
)

booking_summary = (
    trip_load_analysis
    .groupby("booking_type")
    .agg(
        total_trips=("trip_id", "count"),
        trips_with_missing_assignment=("any_assignment_missing", "sum")
    )
)

booking_summary["missing_assignment_rate_pct"] = (
    booking_summary["trips_with_missing_assignment"]
    / booking_summary["total_trips"]
    * 100
)

print("Missing assignment rate by booking type:")
print(booking_summary)

Missing assignment rate by booking type:
              total_trips  trips_with_missing_assignment  \
booking_type                                               
Contract            21545                           1234   
Dedicated           42337                           2511   
Spot                21528                           1207   

              missing_assignment_rate_pct  
booking_type                               
Contract                         5.727547  
Dedicated                        5.930982  
Spot                             5.606652  


In [31]:
# ============================================================
# MISSING ASSIGNMENT RATE BY LOAD TYPE
# ============================================================

load_type_summary = (
    trip_load_analysis
    .groupby("load_type")
    .agg(
        total_trips=("trip_id", "count"),
        trips_with_missing_assignment=("any_assignment_missing", "sum")
    )
)

load_type_summary["missing_assignment_rate_pct"] = (
    load_type_summary["trips_with_missing_assignment"]
    / load_type_summary["total_trips"]
    * 100
)

print("Missing assignment rate by load type:")
print(load_type_summary)

Missing assignment rate by load type:
              total_trips  trips_with_missing_assignment  \
load_type                                                  
Dry Van             42464                           2463   
Refrigerated        42946                           2489   

              missing_assignment_rate_pct  
load_type                                  
Dry Van                          5.800207  
Refrigerated                     5.795650  


### Investigation Findings

The automated audit identified missing assignment identifiers in the
`trips` table:

- `driver_id`: 1,714 missing values
- `truck_id`: 1,672 missing values
- `trailer_id`: 1,680 missing values

Missingness-pattern analysis showed that:

- 80,458 trips have all three assignments present.
- Most incomplete trips have only one missing assignment.
- No trip has all three assignment identifiers missing.

The missing assignment rates were then compared across operational
dimensions.

#### Booking Type

| Booking Type | Missing Assignment Rate |
|--------------|-------------------------:|
| Contract | 5.73% |
| Dedicated | 5.93% |
| Spot | 5.61% |

The rates are relatively close, with only a 0.32 percentage-point
difference between the highest and lowest groups.

#### Load Type

| Load Type | Missing Assignment Rate |
|-----------|-------------------------:|
| Dry Van | 5.80% |
| Refrigerated | 5.80% |

The rates are effectively identical.

### Final Classification

**Data Completeness Concern**

The trips table contains incomplete driver, truck, and trailer
assignments. The missingness does not appear strongly concentrated
within either load type or booking type based on the observed rates.

The available evidence therefore supports treating the issue as a
general assignment-completeness limitation rather than attributing it
to a particular operational segment.

### Data Treatment

No identifiers will be artificially inferred or imputed.

The raw `trips.csv` and `loads.csv` files remain unchanged.

### Downstream BI Consideration

Trip-level KPIs involving driver, truck, or trailer attribution must
account for incomplete assignment coverage.

Analyses grouped by these dimensions may exclude or underrepresent
approximately 5.8% of trips.

The assignment completeness limitation should be documented in the
data model and KPI definitions before dashboard development.

# 5. Final Dataset Audit Summary

## Audit Objective

The dataset audit was performed to understand the structural,
quality, completeness, and analytical limitations of the raw
Logistics Operations dataset before beginning formal data validation.

The audit consisted of automated profiling followed by targeted
business and cross-table investigations.

## Final Classification

### Valid Business Conditions

1. Driver `termination_date` is missing for active drivers and
   populated for terminated drivers. This is consistent with
   employment status.

2. Negative facility longitude values are valid because the dataset
   represents locations in the United States, where western
   longitudes are negative.

### Dataset Limitations

1. `drivers.cdl_class` contains only `A`.
2. `loads.load_status` contains only `Completed`.
3. `trailers.length_feet` contains only `53`.
4. `trailers.status` contains only `Active`.
5. `trips.trip_status` contains only `Completed`.
6. `trucks.fuel_type` contains only `Diesel`.

These are limitations of the available dataset rather than confirmed
data-quality errors.

### Data Completeness / Assignment Concerns

1. `fuel_purchases.truck_id` contains 3,880 missing values.
2. `fuel_purchases.driver_id` contains 3,988 missing values.
3. `safety_incidents.truck_id` contains 1 missing value.
4. `safety_incidents.driver_id` contains 1 missing value.
5. `trips.driver_id` contains 1,714 missing values.
6. `trips.truck_id` contains 1,672 missing values.
7. `trips.trailer_id` contains 1,680 missing values.

Cross-table investigation showed that missing fuel and incident
assignments cannot be recovered through the available `trip_id`
relationships.

Trip assignment missingness is approximately 5.8% and is not
meaningfully concentrated by `load_type` or `booking_type` based on
the observed rates.

## Data Treatment Decision

The raw source files will remain immutable.

No missing identifiers will be artificially inferred or imputed
during the audit.

Any transformation or treatment required for Power BI will be
handled later during the Data Validation and BI Engineering stages.

## Stage 1 Decision

The dataset is suitable to proceed to formal Data Validation.

The identified limitations and completeness concerns must be carried
forward into:

- Data model design
- KPI definitions
- Power Query transformations
- DAX measures
- Dashboard interpretation
- Final project documentation

In [32]:
# ============================================================
# DATASET AUDIT COMPLETION
# ============================================================

print("=" * 70)
print("STAGE 1 - DATASET AUDIT COMPLETE")
print("=" * 70)

print("\nFinal decision:")
print("Dataset is suitable to proceed to Stage 2 - Data Validation.")

print("\nImportant constraints identified:")
print("- Dataset limitations documented")
print("- Assignment completeness concerns documented")
print("- No artificial identifier imputation")
print("- Raw source files remain immutable")

print("\nNext stage:")
print("Stage 2 - Data Validation")

STAGE 1 - DATASET AUDIT COMPLETE

Final decision:
Dataset is suitable to proceed to Stage 2 - Data Validation.

Important constraints identified:
- Dataset limitations documented
- Assignment completeness concerns documented
- No artificial identifier imputation
- Raw source files remain immutable

Next stage:
Stage 2 - Data Validation
